**Problem 4**: So far, we've assumed that the wheels can instantly produce whatever velocity or torque our controller commands. In reality, the controller can only command the motor voltage, while the motor's current, torque, and speed evolve according to its electrical and mechanical dynamics. How can we control the motor from the voltage input?

First, we model our DC motor. Applying Kirchoff's law around the armature yields:
$$
V = Ri + L \dfrac{di}{dt} + K_e \omega
$$
where:
- $V$ is voltage
- $R$ is the resistance of the armature
- $L$ is the inductance of the armature
- $K_e$ is the back electromagnetic force (EMF) constant

This is the electrodynamics of the system. But we still need to connect it to the mechanical model:
$$
J \dot{\omega} = \tau - b \omega - \tau_{load}
$$
In fact, we also have:
$$ e = K_e \omega$$
where $e$ is the electromotive force, and:
$$ \tau = K_t i $$
where $K_t$ is the torque constant. What these two equations say is basically: the electromotive force generated by a DC motor is proportional to the angular velocity, and the torque of the motor is proportional to the current going through it. For an ideal motor with no losses, we also have:
$$ P_{elec} = P_{mech}$$
by conservation of energy. But $P_{elec} = ei$, and $P_{mech} = \tau \omega$ so it means that for an ideal motor described in SI units:
$$ K_e = K_t$$
Now there are two ways for us to describe and analyse the system. We'll use both:

**1. State space representation**

A state is the minimum set of variables that completely describes the system at the current time, allowing its future behaviour to be predicted given the input. Typically, we write:
$$ \dot{\mathbf{x}} = \mathbf{Ax} + \mathbf{Bu}$$
where $x$ is the state vector, $u$ is the control input, and $A$, $B$ are coefficient matrices. So, $\dot{x}$ tells you how the system is changing at every step. For programming, we can discretize the system using the equation:
$$\mathbf{{x_{k+1}}} = \mathbf{{x_k}} + (\mathbf{Ax_k} + \mathbf{Bu_k})\Delta t$$
For our DC motor model, we can choose:
$$ \mathbf{x} = \begin{bmatrix} i \\ \omega \end{bmatrix} \hspace{40pt} \mathbf{u} = V$$
Then our dynamics would be:
$$\dot{i} = \dfrac{1}{L} (V - Ri - K_e \omega)$$
$$\dot{\omega} = \dfrac{1}{J} ( K_t i - b \omega - \tau_{load})$$
In matrix form, that is:
$$
\begin{bmatrix} \dot{i} \\ \dot{\omega} \end{bmatrix} = \begin{bmatrix} -\dfrac{R}{L} & -\dfrac{K_e}{L} \\ \dfrac{K_t}{J} & -\dfrac{b}{J} \end{bmatrix} \begin{bmatrix} i \\ \omega \end{bmatrix} + \begin{bmatrix} \dfrac{1}{L} \\ 0 \end{bmatrix} V
$$
We approximate the armature inductance as negligible. Inductance resists the change in current, and often times, the electrical dynamics settle far faster than the mechanical dynamics. The electrical time constant is $\tau = \dfrac{L}{R}$. Because the electrical time constant is usually much smaller than the mechanical time constant, current dynamics settle much faster than the motor speed dynamics. Therefore, we can approximate the current as reaching its steady-state value instantly.
$$
i = \dfrac{V - K_e \omega}{R}
$$
Given the motor speed, any applied voltage determines the armature current, which in turn determines the generated torque. We get a direct understanding of how the internal states evolve over time: voltage affects current, current produces torque, and torque changes speed. 

Basically, we arrived at the fact that voltage does not directly control speed. It first changes current, which produces torque, which then accelerates the motor. To analyse how the output change directly from information of the input, we can use **transfer functions**.

**2. Transfer functions**

A transfer function describes how an output of a system respond, given the input. Normally it's written as:
$$ G(s) = \dfrac{\text{output(s)}}{\text{input(s)}} $$
The Laplace transform converts time-domain differential equations into equations involving the complex variable $s$, where differentiation becomes multiplication by $s$. This allows us to analyze the system using algebra instead of differential equations.

Now let's find the transfer function for our DC motor. We want to find how the angular velocity changes as the voltage change, so we define:
$$ G(s) = \dfrac{\Omega(s)}{V(s)}$$

Knowing the dynamics from above, we will take the Laplace transform of their ODEs. It transforms derivatives into multiplication by $s$ and integrals into division by $s$. We arrive at:
$$ V(s) = RI(s) + LsI(s) + K_e \Omega(s)$$
$$ Js\Omega(s) + b\Omega(s) = K_t I(s)$$
The rest is simple algebra. We solve for current:
$$ I(s) = \dfrac{Js + b}{K_t} \Omega(s)$$
Plugging back in to the voltage equation yields:
$$ V(s) = \Omega(s) \left[ \dfrac{(R+Ls)(Js + b) + K_e K_t}{K_t} \right]$$
Dividing both sides and assuming $L \approx 0$ yields the transfer function:
$$ G(s) = \dfrac{\Omega(s)}{V(s)} = \dfrac{K_t}{RJs + Rb + K_t K_e}$$
Because voltage is a signal, we can plug in a signal $V(s)$, take the inverse Laplace transform of the system to move back to the time domain and see exactly how the system behaves over time. Instead of only understanding the instantaneous relationship between voltage, current, torque, and speed, we now have a complete input-output model that predicts how the motor responds to any input signal over time.

It is important to stress that transfer functions only describe linear time-invariant (LTI) systems. The Laplace transform itself can be applied to nonlinear systems, but nonlinear systems cannot generally be represented by a single transfer function.. A state-space representation may be represented nonlinearly with $$\dot{x} = f(x, u, t)$$ and then linearised around operating points, where linear control works better.

Now let's see exactly how transfer functions let us know system dynamics from an input voltage signal. Let's say we give the motor a constant voltage $V(t) = V_0$. Taking the Laplace transform yields:
$$V(s) = \dfrac{V_0}{s}$$
From our transfer function:
$$\Omega(s) = \dfrac{K_t}{RJs + Rb + K_t K_e} . \dfrac{V_0}{s}$$
Now we define:
$$\tau = \dfrac{RJ}{Rb + K_e K_t} \hspace{50pt} K = \dfrac{K_t}{Rb + K_e K_t}$$
Plugging this back into the formula for angular velocity yields:
$$\Omega(s) = \dfrac{K V_0}{s(\tau s + 1)}$$
Now comes the interesting part: taking the inverse Laplace transform, we arrive at a close form formula for angular velocity over time:
$$\omega (t) = KV_0 (1 - e^{-\frac{t}{\tau}})$$

In general, for first order systems, defining $K$ and $\tau$ like that is powerful. With the form 
$$G(s) = \dfrac{K}{\tau s + 1}$$
we know a lot of information from the parameters:
- $K$ is called **the DC gain**, telling the final steady state speed for a given voltage
- $\tau$ the **time constant** of the system, telling how quickly the motor reaches the final speed

Normally $\tau$ is on the order of miliseconds: the system settles very quickly, and it reaches $K V_0$ $\text{rad /s}$. After $\tau$ we reaches $1-e^{-1}$ of the steady state speed. After $5\tau$ the system is generally considered to have settled.

But we don't always want full throttle, as seen in the examples with bang bang controllers. We need to find a way to reach any velocity from voltage alone.

To do that, motor driver circuit boards use Pulse-Width-Modulation (PWM). Using a high frequency square wave with varying duty cycles, we can replicate the behaviour of lower voltage by rapidly switching on and off the circuit. Mathematically, a PWM signal is described as:
$$
V(t) = 
\begin{cases}
V_0, \hspace{10pt} & 0 \leq t \mod T < DT\\
0, \hspace{10pt} &DT \leq t \mod T < T
\end{cases}
$$
where:
- $V_0$ is the supply voltage
- $T$ is the PWM period
- $D \in [0;1]$ is the duty cycle

The key is that we make the PWM cycle run at a high frequency such that $T \ll \tau$. Then, $e^{-\frac{T}{\tau}} \approx 1$. This means that the speed does not change as much through each duty cycle: The motor's electrical and mechanical dynamics act together as a low-pass filter, blocking this rapid signal. Because of that, only the low-frequency (average) component of the PWM signal significantly affects the motor speed.

Strictly speaking, the motor experiences the full supply voltage during the "on" portion of each PWM cycle. The average voltage model is valid because the PWM frequency is much higher than the motor bandwidth.
$$V_{avg} = \dfrac{1}{T} \int_0^T V(t)dt = \dfrac{1}{T} V_0 DT = DV_0 $$
Consequently, its speed is almost identical to the response that would be obtained if a constant voltage equal to the average PWM voltage were applied:
$$V_{avg} = D V_0 $$
By controlling the duty cycle, which is the amount of time the signal stays on in a cycle, we can command the voltage coming to the motor. 

However, the entire derivation assumes no friction, no backlash, and generally no imperfections. Real motors have static friction, Coulomb friction, and a dead zone near zero voltage. Because of that, we still need controllers to control the motor well.

All of that goes to show that, in a real system, there are many layers of dynamics working with each other. To achieve better precision, engineers can control each output separately. For example, in an industrial servo application, one might see:
- A 20kHz PID loop handling the electrodynamics
- A 1kHz PID loop handling velocity control
- A 50Hz PID loop handling position control
- A dynamic feedforward term for gravity compensation
- An S-curve motion profile for smooth motion

These PID loops all hinge on the fact that the time constant of each dynamic system are different, and assume that the faster dynamic settle instantly compared to the slower one. This is called **cascading PID**. We'll try to implement and tune it. For simplicity's sake, we will let all three loops be running concurrently (so that the plot doesn't take minutes to render).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from disturbance import HalfSineDisturbance
from feedback import PID
from motion_profile import SCurveProfile

dt = 0.001 # Time step
time = 10.0 # Total simulation time
t = np.arange(0, time, dt)

# System state
theta = np.zeros_like(t)
omega = np.zeros_like(t)

load = np.zeros_like(t) # disturbance torque
accel = np.zeros_like(t) # angular acceleration

u = np.zeros_like(t)      # control input
pid = np.zeros_like(t)      # PID control input
ff = np.zeros_like(t)      # Feedforward control input

theta[0] = np.pi / 2 
omega[0] = 0.0
accel[0] = 0.0

# Motor

resistance = 0.4          # winding resistance (Ohm)
inductance = 0.002        # inductance (H)

Kt = 0.5        # torque constant (Nm/A)
Ke = 0.5        # back EMF constant (Vs/rad)

i = np.zeros_like(t)

# Desired position
target = 0

profile = SCurveProfile(start_pos=theta[0], end_pos=target, max_vel=np.pi, max_accel=np.pi * 2, max_jerk = np.pi * 3)

position_pid = PID(
    Kp=10.0,
    Ki=0.01,
    Kd=1.0,
    alpha=0.9,
    u_max=6.0,
    u_min=-6.0
)

velocity_pid = PID(
    Kp=5.0,
    Ki=0.01,
    Kd=0.05,
    alpha=0.9,
    u_max=15.0,
    u_min=-15.0
)

current_pid = PID(
    Kp=3.6,
    Ki=1.0,
    Kd=0.0,
    alpha=0.9,
    u_max=24.0,
    u_min=-24.0
)

m = 0.2  # Mass of the arm
length = 0.5  # Length of the arm
g = 9.81  # Acceleration due to gravity
b = 0.1  # Damping coefficient
inertia = (1/3) * m * length**2  # Moment of inertia

disturbance = HalfSineDisturbance(shot_rate=1.0, shot_duration=0.4, shot_torque=2.0, bidirectional=True)

# Simulation
for k in range(len(t)-1):
    t_now = t[k]

    theta_ref = profile.position(t_now)
    omega_ref = profile.velocity(t_now)
    alpha_ref = profile.acceleration(t_now)

    ambient_disturbance = np.random.normal(0, 0.05)
    load[k] = disturbance.update(dt)   

    # Position loop -> velocity command
    omega_correction = position_pid.control(
        current_value=theta[k],
        current_velocity=omega[k],
        dt=dt,
        setpoint=theta_ref,
    )
    omega_cmd = omega_correction + omega_ref   

    # Velocity loop -> current command
    current_correction = velocity_pid.control(
        current_value=omega[k],
        current_velocity=accel[k],
        dt=dt,
        setpoint=omega_cmd,
    )

    current_ff = (inertia * alpha_ref + b * omega_ref + m * g * (length / 2) * np.cos(theta_ref)) / Kt

    current_cmd = np.clip(current_correction + current_ff, -24.0, 24.0)  

    # Current loop -> voltage
    voltage_ff = resistance * current_cmd + Ke * omega_ref

    voltage = current_pid.control(
        current_value=i[k],
        current_velocity=0.0,
        dt=dt,
        setpoint=current_cmd,
    ) + voltage_ff

    di_dt = (voltage - resistance * i[k] - Ke * omega[k]) / inductance
    i[k+1] = i[k] + di_dt * dt

    motor_torque = Kt * i[k]

    theta_ddot = (motor_torque - b * omega[k] - m * g * (length/2) * np.cos(theta[k]) + ambient_disturbance + load[k]) / inertia

    omega[k+1] = omega[k] + theta_ddot * dt
    theta[k+1] = theta[k] + omega[k+1] * dt
    accel[k+1] = theta_ddot

    # telemetry
    u[k] = voltage
    pid[k] = current_correction
    ff[k] = current_ff

u[-1], pid[-1], ff[-1] = u[-2], pid[-2], ff[-2]

# Plotting

fig, (ax1, ax2, ax3, ax4, ax5, ax6) = plt.subplots(6, 1, figsize=(18, 14), sharex=True)

ax1.plot(t, theta)
ax1.axhline(target, color='r', linestyle='--')
ax1.set_ylabel("Angular Position (rad)")
ax1.set_xlabel("Time (s)")

ax2.step(t, u, where='post')
ax2.set_ylabel("Control Input (N·m)")
ax2.set_xlabel("Time (s)")

ax3.step(t, load, where='post')
ax3.set_ylabel("Disturbance Torque (N·m)")
ax3.set_xlabel("Time (s)")

ax4.plot(t, omega)
ax4.set_ylabel("Angular Velocity (rad/s)")
ax4.set_xlabel("Time (s)")

ax5.plot(t, accel)
ax5.set_ylabel("Angular Acceleration (rad/s²)")
ax5.set_xlabel("Time (s)")

ax6.plot(t, i)
ax6.set_ylabel("Current (A)")
ax6.set_xlabel("Time (s)")

plt.show()

Running this simulation, we can see the three loops working in concert: the current loop settles within a few milliseconds, tracking its commanded current almost perfectly thanks to the voltage feedforward canceling out resistive and back-EMF terms in real time; the velocity loop, an order of magnitude slower, shapes the arm's angular rate to follow the S-curve profile's velocity trace even as gravity and periodic disturbance shots try to knock it off course; and the position loop, slower still, quietly corrects any residual drift the inner two loops accumulate. Notice how little work each individual PID actually has to do — most of the heavy lifting is already done by feedforward, with feedback only patching over what the model doesn't capture: unmodeled friction, the disturbance torque, and the electrical dynamics we approximated away when deriving our transfer function. This is the payoff of cascading: rather than one controller trying to reason about position, velocity, and current all at once, we decompose the problem into three simpler ones, each operating on a timescale where the layer beneath it can be trusted to have "already happened." 

Still, tuning the controller is a big problem. If the dynamics change: different arm model, different payload, different actuators, tuning the control parameters by the graph alone can be difficult. We'll need a way to analyse the behaviour of the plant under different conditions. But for now: **Problem solved**.